# 04 関係状態の時間変化

1 年 = 1 ステップで、2 人の満足度と関係の状態を更新します（**状態遷移・時系列シミュレーション**）。
F3 では解消確率を外から与えましたが、F4 では**関係状態（満足度）とストレスから毎年計算**します。
計算：`relationship_dynamics.py`

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 1. 1 年の流れ

1. その年の生活条件（ストレスイベントを反映）
2. 戦略が分担 $(x, y)$ を決める
3. その年の満足度（効用）$U_i(t)$
4. 満足度の更新：$S_i(t) = \mathrm{clip}\big((1-\alpha)\,S_i(t-1) + \alpha\,U_i(t),\ 0,\ 1\big)$
5. 関係全体の満足度：$R(t) = (1-m)\cdot\text{平均}(S_A, S_B) + m\cdot\text{最低}(S_A, S_B)$
6. 関係解消確率：$p(t) = \mathrm{clip}\big(\text{baseline} + \beta\,(1 - R(t)) + \gamma\,\text{Stress}(t),\ 0,\ 1\big)$
7. 一様乱数 $u < p(t)$ なら解消（Core では再開しない。以後の更新は止める）

$\alpha$・$m$・baseline・$\beta$・$\gamma$ はすべて**教育用の架空の値**です。

In [ ]:
print({k: params[k] for k in ("alpha", "relationship_min_weight", "baseline_hazard", "beta", "gamma", "base_stress", "stress_duration")})
for e in events["combined"]:
    print(f"Year {e['year']}: {e['note']}（ストレス +{e['stress']}）")

## 2. 年数と満足度

Adaptive 戦略で、関係が続いた場合の満足度の推移です（赤い点線 = ストレスイベント）。

In [ ]:
#@title 年数と満足度
EVENTS = events["combined"]
EV_YEARS = rd.event_years(EVENTS)
recs = rd.trajectory(SC, "adaptive", params, EVENTS)
show(pc.plot_satisfaction_timeseries(recs, "Satisfaction over time - Adaptive, if the relationship continues (model)",
                                     EV_YEARS), "04_satisfaction_timeseries")

## 3. 年数と関係解消確率

In [ ]:
#@title 年数と関係解消確率
trajs = {s: rd.trajectory(SC, s, params, EVENTS) for s in ds.STRATEGIES}
show(pc.plot_yearly(trajs, "dissolution_probability", "annual dissolution probability p(t)",
                    "Annual dissolution probability by strategy (model)", EV_YEARS), "04_hazard")

## 4. 1 回のシミュレーション（確率的な判定）

同じ条件でも、**乱数によって解消する年が変わります**。1 回の結果だけで判断しないでください。

In [ ]:
#@title 1 回のシミュレーション
for rep in range(5):
    out = rd.simulate(SC, "adaptive", params, EVENTS, replicate=rep)
    state = f"Year {out['dissolution_year']} に解消" if out["dissolved"] else f"{out['observed_years']} 年まで継続（打ち切り）"
    print(f"反復 {rep}: {state}")
# イベントの後に解消した 1 回を探して描く（線形探索）
out = next(o for o in (rd.simulate(SC, "adaptive", params, EVENTS, replicate=r) for r in range(200))
           if o["dissolved"] and o["dissolution_year"] > max(EV_YEARS))
show(pc.plot_satisfaction_timeseries(out["records"], f"One run that ended in dissolution (replicate {out['replicate']}, model)",
                                     EV_YEARS, YEARS), "04_one_run")

## 5. ストレスイベント前後の満足度

Fixed と Adaptive について、イベントなし／ありの関係満足度を比べます。

In [ ]:
#@title ストレスイベント前後の満足度
series = {"Fixed, no event": rd.trajectory(SC, "fixed", params, ()),
          "Fixed, events": rd.trajectory(SC, "fixed", params, EVENTS),
          "Adaptive, no event": rd.trajectory(SC, "adaptive", params, ()),
          "Adaptive, events": rd.trajectory(SC, "adaptive", params, EVENTS)}
colors = {"Fixed, no event": pc.RAMP[0], "Fixed, events": pc.STRATEGY_COLORS["fixed"],
          "Adaptive, no event": "#f5c86a", "Adaptive, events": pc.STRATEGY_COLORS["adaptive"]}
show(pc.plot_yearly(series, "relationship_satisfaction", "relationship satisfaction R(t)",
                    "Relationship satisfaction with and without stress events (model)", EV_YEARS, (0, 1), colors),
     "04_stress_before_after")
for name in ("Fixed, events", "Adaptive, events"):
    for y in EV_YEARS:
        r = ev.stress_response(series[name], y)
        print(f"{name} / Year {y}: 前 2 年 {r['before']:.3f} → 後 2 年 {r['after']:.3f}（{r['change']:+.3f}）")

## 6. 関係継続率の曲線

5 シード × 200 回の反復から求めた継続率（実線）と、解消確率から計算した理論値 $\prod_t (1 - p(t))$（点線）です。
5・10・20 年に縦線を引いています。

In [ ]:
#@title 関係継続率の曲線
res = ev.compare_strategies(SC, params, EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
show(pc.plot_survival({s: res[s]["survival"] for s in ds.STRATEGIES},
                      "Continuation rate by strategy (model, 1,000 runs each)",
                      theory={s: res[s]["theoretical_survival"] for s in ds.STRATEGIES}), "04_survival")
table([{"name": ds.LABELS[s], **res[s]} for s in ds.STRATEGIES],
      [("name", "strategy", "{}"), ("continuation_5", "5Y", "{:.1%}"), ("continuation_10", "10Y", "{:.1%}"),
       ("continuation_10_sd", "10Y sd", "{:.1%}"), ("continuation_20", "20Y", "{:.1%}"),
       ("dissolution_rate", "dissolved", "{:.1%}")])

## 7. 継続期間

観測期間（20 年）の終わりまで続いた関係は「**打ち切り**」です。20 年で解消したのではありません（その後は分かりません）。

In [ ]:
#@title 継続期間の分布
mc = rd.monte_carlo(SC, "adaptive", params, EVENTS, 1000, seed=params["random_seed"])
show(pc.plot_duration_hist(mc["outcomes"], YEARS, "Year of dissolution - Adaptive (model, 1,000 runs)"), "04_duration")
print(ev.duration_summary(mc["outcomes"]))

## 8. 学生向け TODO：パラメータとストレスイベントを変える

In [ ]:
# TODO 1：満足度の更新速度 alpha を変える
curves = {}
for a in [0.1, 0.3, 0.6]:
    p2 = rd.with_changes(params, alpha=a)
    r = ev.compare_strategies(SC, p2, EVENTS, ["fixed"], REPLICATES, SEEDS)["fixed"]
    curves[f"alpha={a}"] = r["survival"]
show(pc.plot_survival(curves, "Fixed strategy: continuation for different alpha (model)"), "04_todo_alpha")

In [ ]:
# TODO 2：ストレスイベントを変える（年・種類・大きさ）
my_events = [{"year": 3, "type": "work_hours_change", "partner": "A", "change": 8, "stress": 0.3, "note": "A の労働時間 +8"},
             {"year": 10, "type": "expense_change", "change": 0.2, "duration": 5, "stress": 0.1, "note": "生活費 +20%（5 年間）"}]
# TODO 3：基準解消確率 baseline_hazard を変える
p3 = rd.with_changes(params, baseline_hazard=0.02)
r = ev.compare_strategies(SC, p3, my_events, ds.STRATEGIES, REPLICATES, SEEDS)
# TODO 4：継続率を比べる
for s in ds.STRATEGIES:
    print(f"{ds.LABELS[s]:15}: 10 年 {r[s]['continuation_10']:.1%} ± {r[s]['continuation_10_sd']:.1%}"
          f"  20 年 {r[s]['continuation_20']:.1%}")

## 考察（チームで記入）

- **TODO**：alpha が大きい（今年の条件にすぐ反応する）と、ストレスイベントの影響はどう変わるか
- **TODO**：継続率の差は、シード間のばらつき（sd）と比べて大きいか
- **TODO**：「打ち切り」を「20 年で解消」と数えると、何が間違うか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）